In [ ]:
from main.frp_utils.aoi_helpers import get_AOI_size_per_snippet
from main.frp_utils.generate_snippets_utils import *

In [ ]:
# Paths to use for annotated snippets
code_snippets_with_data_driven_aois_dir = f'../code_snippets_with_data_driven_aois'
AOI_SIZE_PATH = f'{code_snippets_with_data_driven_aois_dir}/aoi_sizes.csv'

ANNOTATED_SNIPPETS_FOLDERS = {
    'SOURCE_FOLDER': code_snippets_with_data_driven_aois_dir,
    'GENERATOR_FOLDER': f'{code_snippets_with_data_driven_aois_dir}/JSON',
    'SNIPPET_FOLDER': f'{code_snippets_with_data_driven_aois_dir}/Pictures',
    'SUBFOLDER': ['v0', 'v1', 'v2']  # sub folders in source folder
}

# Create folders for results
os.makedirs(ANNOTATED_SNIPPETS_FOLDERS['GENERATOR_FOLDER'], exist_ok=True)
os.makedirs(ANNOTATED_SNIPPETS_FOLDERS['SNIPPET_FOLDER'], exist_ok=True)
for folder in ANNOTATED_SNIPPETS_FOLDERS['SUBFOLDER']:
    os.makedirs(f'{ANNOTATED_SNIPPETS_FOLDERS["GENERATOR_FOLDER"]}/{folder}', exist_ok=True)
    os.makedirs(f'{ANNOTATED_SNIPPETS_FOLDERS["SNIPPET_FOLDER"]}/{folder}', exist_ok=True)

# Patterns to manipulate spaces in code
LEADING_SPACES_PATTERN = re.compile(r'^(?:\[aoi=[^A-Z]\])?( *)')
SINGLE_SPACE_PATTERN = re.compile('([^ ]) ([^ ])')
SINGLE_SPACE_REPLACEMENT = r'\1  \2'

# fixed image sizes
IMAGE_SIZE = (1144, 720)

In [ ]:
data_template = {
    'font-normal': 'JetBrainsMono-Regular.ttf',
    'font-bold': 'JetBrainsMono-Bold.ttf',
    'font-italic': 'JetBrainsMono-BoldItalic.ttf',
    'font-bold-italic': 'JetBrainsMono-Italic.ttf',
    'font-size': 30,
    'width-margin': 5,  # buffer in width
    'height-margin': 5,  # buffer in height
    'spacing': 35,  # distance between code lines
    'background-color': [255, 255, 255],  # White
    'style': 'vs',
}

In [ ]:
AOI_NAME_COLOR_MAPPING = {
    "1": "#ff0000",  # red
    "2": "#ffa000",  # bright orange
    "3": "#fff000",  # bright yellow"
    "4": "#00ff00",  # green
    "5": "#0000ff",  # blue
    "6": "#ff00ff",  # magenta
    "7": "#00ffff",  # cyan
    "8": "#000000",  # black
    "9": "#808080",  # gray
}

AoiC, WordC = create_images_from_code(ANNOTATED_SNIPPETS_FOLDERS["SOURCE_FOLDER"],
                                      ANNOTATED_SNIPPETS_FOLDERS["GENERATOR_FOLDER"],
                                      ANNOTATED_SNIPPETS_FOLDERS["SNIPPET_FOLDER"],
                                      ANNOTATED_SNIPPETS_FOLDERS["SUBFOLDER"],
                                      aoi_colors=AOI_NAME_COLOR_MAPPING,
                                      )
#print(AoiC)

In [ ]:
offset_x_direction = 555
offset_y_direction = 271
scale_factor_x = 810 / 1144
scale_factor_y = 540 / 720


def convert_aois_format(all_aois):
    converted = {}
    for snippet, coords_list in all_aois.items():
        aois_dict = {}
        for idx, (left, top, right, bottom) in enumerate(coords_list, start=1):
            aois_dict[str(idx)] = {
                'left': int(scale_factor_x * left + offset_x_direction),
                'top': int(scale_factor_y * top + offset_y_direction),
                'right': int(scale_factor_x * right + offset_x_direction),
                'bottom': int(scale_factor_y * bottom + offset_y_direction),
            }
        converted[snippet] = aois_dict
    return converted


# Convert the AOIs to the desired format
all_aois = convert_aois_format(AoiC)
level_area_data = get_AOI_size_per_snippet(all_aois)
level_area_data.to_csv(AOI_SIZE_PATH, sep=';', index=False)
display(level_area_data)